# Notebook 3b — Multi-Seed Training + Variance Statistics

Notebook 3 showed a ~10× variance collapse under leakage, but from a single run that number is unstable. This notebook retrains across several seeds so you can make a **defensible statistical claim** instead of quoting one fragile ratio.

It does two jobs at once:
1. **Locks the variance statistics** — per-seed fold-std for each protocol, plus a formal test (RQ2).
2. **Produces the seed models your stability experiment needs** — same architecture, different seeds, saved for Notebook 6 (RQ3).

**Cost:** `n_seeds × 5 folds × 2 splits` runs. 

In [1]:
# Cell 1 — imports
!pip install timm --quiet

from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import timm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy import stats

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU. Turn on the T4 accelerator.")

device: cuda


## Cell 2 — settings

In [2]:
# Cell 2 — the only cell you edit
IN_DIR  = "/kaggle/input"
OUT_DIR = "/kaggle/working/runs_multiseed"

ARCH       = "resnet18"
SEEDS      = [42, 1, 2]       
FOLDS      = [0, 1, 2, 3, 4]
EPOCHS     = 20
BATCH_SIZE = 32
LR         = 3e-4

In [3]:
# Cell 3 — locate inputs
def find(name):
    hits = list(Path(IN_DIR).rglob(name))
    assert hits, f"{name} not found under {IN_DIR}."
    return hits[0]

images   = np.load(find("images.npy"))
manifest = pd.read_csv(find("manifest_splits.csv"))
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print("images", images.shape, "| slices", len(manifest))

images (3064, 224, 224) | slices 3064


## Cell 4 — dataset and reproducible seeding

The seeding here is thorough on purpose: numpy, torch, CUDA, **and** the DataLoader worker/shuffle. Without seeding the workers, the augmentation flips would be random regardless of seed, and your "different seeds" wouldn't be reproducible.

In [4]:
# Cell 4 — dataset + seeding
IMAGENET_MEAN = np.array([0.485,0.456,0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229,0.224,0.225], dtype=np.float32)

class SliceDataset(Dataset):
    def __init__(self, images, labels, train=False):
        self.images, self.labels, self.train = images, labels, train
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        img = self.images[i].astype(np.float32)/255.0
        if self.train and np.random.rand() < 0.5:
            img = np.fliplr(img).copy()
        img = np.stack([img]*3, axis=-1)
        img = (img - IMAGENET_MEAN)/IMAGENET_STD
        return torch.from_numpy(img.transpose(2,0,1)), int(self.labels[i])

def set_seed(seed):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id):
    ws = torch.initial_seed() % 2**32
    np.random.seed(ws)

print("dataset + seeding defined")

dataset + seeding defined


## Cell 5 — train / eval helpers

In [5]:
# Cell 5 — helpers
def run_epoch(model, loader, criterion, optimizer, train):
    model.train() if train else model.eval()
    total, probs_all, true_all = 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x); loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item()*len(y)
            probs_all.append(torch.softmax(logits,1).detach().cpu().numpy())
            true_all.append(y.cpu().numpy())
    probs = np.concatenate(probs_all); true = np.concatenate(true_all)
    return total/len(true), probs, true

def score(true, probs):
    pred = probs.argmax(1)
    return {"accuracy": accuracy_score(true,pred),
            "macro_f1": f1_score(true,pred,average="macro"),
            "auc": roc_auc_score(true,probs,multi_class="ovr",average="macro")}
print("helpers defined")

helpers defined


## Cell 6 — train one (seed-aware)

In [6]:
# Cell 6 — train one
def train_one(seed, split_col, fold):
    set_seed(seed)
    test_mask = (manifest[split_col] == fold).values
    tr_idx, te_idx = np.where(~test_mask)[0], np.where(test_mask)[0]
    labels = manifest["label"].values - 1

    g = torch.Generator(); g.manual_seed(seed)
    tr_dl = DataLoader(SliceDataset(images[tr_idx], labels[tr_idx], True),
                       batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                       worker_init_fn=seed_worker, generator=g)
    te_dl = DataLoader(SliceDataset(images[te_idx], labels[te_idx], False),
                       batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = timm.create_model(ARCH, pretrained=True, num_classes=3).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    for ep in range(EPOCHS):
        run_epoch(model, tr_dl, criterion, optimizer, True); scheduler.step()

    _, probs, true = run_epoch(model, te_dl, criterion, None, False)
    m = score(true, probs)
    tag = f"{ARCH}_seed{seed}_{split_col}_fold{fold}"
    torch.save(model.state_dict(), Path(OUT_DIR)/f"model_{tag}.pt")
    np.savez(Path(OUT_DIR)/f"preds_{tag}.npz", probs=probs, true=true, test_idx=te_idx)
    return m
print("train_one defined")

train_one defined


## Cell 7 — run the grid

In [7]:
# Cell 7 — run everything
rows = []
total = len(SEEDS)*len(["fold_honest","fold_leaky"])*len(FOLDS)
k = 0
for seed in SEEDS:
    for split_col in ["fold_honest", "fold_leaky"]:
        for fold in FOLDS:
            k += 1
            m = train_one(seed, split_col, fold)
            print(f"[{k:2d}/{total}] seed {seed} {split_col} fold {fold} "
                  f"-> acc {m['accuracy']:.4f}")
            rows.append({"arch": ARCH, "seed": seed, "split": split_col, "fold": fold, **m})

res = pd.DataFrame(rows)
res.to_csv(Path(OUT_DIR)/"results_multiseed.csv", index=False)
print("\nsaved results_multiseed.csv +", total, "model/pred files")

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

[ 1/30] seed 42 fold_honest fold 0 -> acc 0.9628
[ 2/30] seed 42 fold_honest fold 1 -> acc 0.8843
[ 3/30] seed 42 fold_honest fold 2 -> acc 0.9801
[ 4/30] seed 42 fold_honest fold 3 -> acc 0.9182
[ 5/30] seed 42 fold_honest fold 4 -> acc 0.9346
[ 6/30] seed 42 fold_leaky fold 0 -> acc 0.9804
[ 7/30] seed 42 fold_leaky fold 1 -> acc 0.9918
[ 8/30] seed 42 fold_leaky fold 2 -> acc 0.9821
[ 9/30] seed 42 fold_leaky fold 3 -> acc 0.9821
[10/30] seed 42 fold_leaky fold 4 -> acc 0.9804
[11/30] seed 1 fold_honest fold 0 -> acc 0.9628
[12/30] seed 1 fold_honest fold 1 -> acc 0.9049
[13/30] seed 1 fold_honest fold 2 -> acc 0.9635
[14/30] seed 1 fold_honest fold 3 -> acc 0.9197
[15/30] seed 1 fold_honest fold 4 -> acc 0.9430
[16/30] seed 1 fold_leaky fold 0 -> acc 0.9804
[17/30] seed 1 fold_leaky fold 1 -> acc 0.9837
[18/30] seed 1 fold_leaky fold 2 -> acc 0.9657
[19/30] seed 1 fold_leaky fold 3 -> acc 0.9853
[20/30] seed 1 fold_leaky fold 4 -> acc 0.9755
[21/30] seed 2 fold_honest fold 0 -> acc

## Cell 8 — variance statistics (the point of this notebook)

**Table 1 — per-seed fold spread.** For each seed, the standard deviation across the 5 folds, per protocol. If honest_std is many times larger than leaky_std *for every seed*, the variance collapse is robust, not a fluke. That consistency across seeds is your strongest evidence.

**Table 2 — Levene's test per seed.** Formally tests whether the 5 honest fold accuracies and 5 leaky fold accuracies have equal variance. Within a seed the five folds are disjoint patient groups, so this is a clean test. Small p-values across all seeds = defensible claim of unequal variance.

In [8]:
# Cell 8 — statistics
print("TABLE 1 — fold-to-fold std per seed")
print("-"*52)
tab = []
for seed in SEEDS:
    h = res[(res.seed==seed)&(res.split=="fold_honest")].accuracy.values
    l = res[(res.seed==seed)&(res.split=="fold_leaky")].accuracy.values
    ratio = h.std(ddof=1)/l.std(ddof=1) if l.std(ddof=1)>0 else float("inf")
    tab.append({"seed": seed, "honest_mean": h.mean(), "honest_std": h.std(ddof=1),
                "leaky_mean": l.mean(), "leaky_std": l.std(ddof=1), "std_ratio": ratio})
tab = pd.DataFrame(tab)
print(tab.round(4).to_string(index=False))

print("\nTABLE 2 — Levene test for equal variance (honest vs leaky), per seed")
print("-"*52)
for seed in SEEDS:
    h = res[(res.seed==seed)&(res.split=="fold_honest")].accuracy.values
    l = res[(res.seed==seed)&(res.split=="fold_leaky")].accuracy.values
    W, p = stats.levene(h, l)
    verdict = "variances DIFFER" if p < 0.05 else "no sig. difference"
    print(f"  seed {seed}: W={W:.3f}  p={p:.4f}  -> {verdict}")

print("\nSUMMARY")
print("-"*52)
print(f"Mean honest fold-std : {tab.honest_std.mean():.4f}")
print(f"Mean leaky  fold-std : {tab.leaky_std.mean():.4f}")
print(f"Mean std ratio       : {tab.std_ratio.mean():.1f}x "
      f"(range {tab.std_ratio.min():.1f}-{tab.std_ratio.max():.1f})")

TABLE 1 — fold-to-fold std per seed
----------------------------------------------------
 seed  honest_mean  honest_std  leaky_mean  leaky_std  std_ratio
   42       0.9360      0.0376      0.9834     0.0048     7.8006
    1       0.9388      0.0261      0.9781     0.0079     3.3088
    2       0.9446      0.0358      0.9827     0.0025    14.5256

TABLE 2 — Levene test for equal variance (honest vs leaky), per seed
----------------------------------------------------
  seed 42: W=7.238  p=0.0275  -> variances DIFFER
  seed 1: W=5.093  p=0.0540  -> no sig. difference
  seed 2: W=7.182  p=0.0279  -> variances DIFFER

SUMMARY
----------------------------------------------------
Mean honest fold-std : 0.0332
Mean leaky  fold-std : 0.0051
Mean std ratio       : 8.5x (range 3.3-14.5)


In [9]:
"""

The per-seed Levene tests (5 vs 5) are underpowered. This pools across seeds
to restore power, and adds tests that are more robust than Levene.

"""
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

# If running standalone, set this. In-notebook, `res` already exists.
try:
    res
except NameError:
    RESULTS_CSV = "/kaggle/working/runs_multiseed/results_multiseed.csv"
    res = pd.read_csv(RESULTS_CSV)

honest = res[res.split == "fold_honest"].accuracy.values   # 15 values (3 seeds x 5 folds)
leaky  = res[res.split == "fold_leaky"].accuracy.values     # 15 values

print("n per group:", len(honest), "honest,", len(leaky), "leaky")
print(f"honest: mean {honest.mean():.4f}  std {honest.std(ddof=1):.4f}")
print(f"leaky : mean {leaky.mean():.4f}  std {leaky.std(ddof=1):.4f}")

# --- Test 1: pooled Levene (center='median' = Brown-Forsythe, robust) ---
W, p = stats.levene(honest, leaky, center="median")
print(f"\nPooled Brown-Forsythe (Levene, median): W={W:.3f}  p={p:.5f}")

# --- Test 2: Fligner-Killeen, the most non-normality-robust variance test ---
X, pf = stats.fligner(honest, leaky)
print(f"Fligner-Killeen:                        X={X:.3f}  p={pf:.5f}")

# --- Test 3: paired on per-seed std (unit = seed) ---
seeds = sorted(res.seed.unique())
h_std = [res[(res.seed==s)&(res.split=="fold_honest")].accuracy.std(ddof=1) for s in seeds]
l_std = [res[(res.seed==s)&(res.split=="fold_leaky")].accuracy.std(ddof=1) for s in seeds]
print(f"\nPer-seed std pairs (honest, leaky):")
for s, a, b in zip(seeds, h_std, l_std):
    print(f"  seed {s}: {a:.4f} vs {b:.4f}  -> honest larger: {a > b}")
print(f"Honest std larger in {sum(a>b for a,b in zip(h_std,l_std))}/{len(seeds)} seeds")

# --- Effect size that is stable: mean absolute deviation ratio, with CI by bootstrap ---
def fold_dispersion(vals):
    return np.mean(np.abs(vals - np.mean(vals)))   # MAD, less noisy than std here

rng = np.random.default_rng(0)
ratios = []
for _ in range(5000):
    hb = rng.choice(honest, len(honest), replace=True)
    lb = rng.choice(leaky,  len(leaky),  replace=True)
    d_l = fold_dispersion(lb)
    if d_l > 0:
        ratios.append(fold_dispersion(hb) / d_l)
ratios = np.array(ratios)
print(f"\nDispersion ratio (honest/leaky), bootstrap median "
      f"{np.median(ratios):.1f}, 95% CI [{np.percentile(ratios,2.5):.1f}, "
      f"{np.percentile(ratios,97.5):.1f}]")
print("Report the DIRECTION and the pooled test, not a single point ratio.")

n per group: 15 honest, 15 leaky
honest: mean 0.9398  std 0.0313
leaky : mean 0.9814  std 0.0056

Pooled Brown-Forsythe (Levene, median): W=23.959  p=0.00004
Fligner-Killeen:                        X=13.553  p=0.00023

Per-seed std pairs (honest, leaky):
  seed 1: 0.0261 vs 0.0079  -> honest larger: True
  seed 2: 0.0358 vs 0.0025  -> honest larger: True
  seed 42: 0.0376 vs 0.0048  -> honest larger: True
Honest std larger in 3/3 seeds

Dispersion ratio (honest/leaky), bootstrap median 7.2, 95% CI [3.5, 15.7]
Report the DIRECTION and the pooled test, not a single point ratio.
